In [1]:
import os, json, shutil
from pathlib import Path
import numpy as np
import nibabel as nib
from tqdm import tqdm
import pandas as pd
import torch
from sklearn.model_selection import KFold
import importlib
import nnunetv2 as _nnunetv2_pkg
import torch as _t, numpy as _np

# Fontes RAW oficiais
TRAIN_RAW = Path("/home/henrique.grande/brats_goat_experiments/data/raw/MICCAI2024-BraTS-GoAT-TrainingData-With-GroundTruth")
VAL_RAW   = Path("/home/henrique.grande/brats_goat_experiments/data/raw/MICCAI2024-BraTS-GoAT-ValidationData")

# Base ISOLADA deste experimento (nao sobrescreve outros)
NNUNET_BASE = Path("/home/henrique.grande/brats_goat_experiments/models_configs/nnUnet_base_1fold_labeled_deform_aug_agressive")
NNUNET_RAW = NNUNET_BASE / "nnUNet_raw"
NNUNET_PREP = NNUNET_BASE / "nnUNet_preprocessed"
NNUNET_RES = NNUNET_BASE / "nnUNet_results"
for p in [NNUNET_RAW, NNUNET_PREP, NNUNET_RES]:
    p.mkdir(parents=True, exist_ok=True)

os.environ["nnUNet_raw"] = str(NNUNET_RAW)
os.environ["nnUNet_preprocessed"] = str(NNUNET_PREP)
os.environ["nnUNet_results"] = str(NNUNET_RES)

DATASET_ID = 961
DATASET_NAME = "BraTS-GoAT-base-1fold-labeled-deform-aug-agressive"
DATASET_FOLDER = NNUNET_RAW / f"Dataset{DATASET_ID:03d}_{DATASET_NAME}"
imagesTr = DATASET_FOLDER / "imagesTr"
labelsTr = DATASET_FOLDER / "labelsTr"
imagesVs = DATASET_FOLDER / "imagesVs"
for p in [imagesTr, labelsTr, imagesVs]:
    p.mkdir(parents=True, exist_ok=True)

# Predicoes isoladas (validacao oficial)
PRED_VAL_OFC = Path("/home/henrique.grande/brats_goat_experiments/predictions/nnUnet_base_1fold_labeled_deform_aug_agressive/ofc_validation")
PRED_VAL_OFC.mkdir(parents=True, exist_ok=True)

# Pipeline padrao nnU-Net - configuracao Large (L)
PLANNER    = "nnUNetPlannerResEncL"
PLANS_NAME = "nnUNetResEncUNetLPlans"
TRAINER    = "nnUNetTrainer_deformAug_agressive"   # 1000 ep + augmentation de deformacao
CONFIG     = "3d_fullres"
FOLD       = 0

DATASET_FOLDER

PosixPath('/home/henrique.grande/brats_goat_experiments/models_configs/nnUnet_base_1fold_labeled_deform_aug_agressive/nnUNet_raw/Dataset961_BraTS-GoAT-base-1fold-labeled-deform-aug-agressive')

In [2]:
print("=" * 60)
print("  Informacoes da GPU")
print("=" * 60)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("  CUDA   :", torch.version.cuda, "| PyTorch:", torch.__version__)
    print("  GPU    :", props.name)
    print("  VRAM   :", f"{props.total_memory/1e9:.2f} GB")
else:
    print("  CUDA   : NAO")
print("=" * 60)

  Informacoes da GPU
  CUDA   : 12.1 | PyTorch: 2.5.1+cu121
  GPU    : NVIDIA H100 80GB HBM3
  VRAM   : 85.02 GB


In [3]:
def get_patient_ids(directory):
    if not directory.exists():
        return []
    return sorted([d.name for d in directory.iterdir() if d.is_dir()])

train_ids = get_patient_ids(TRAIN_RAW)     # rotulados oficiais (com GT)
val_ofc_ids = get_patient_ids(VAL_RAW)     # validacao oficial (sem GT, so inferencia)

print(f"Treino rotulado (With-GroundTruth): {len(train_ids)} | existe: {TRAIN_RAW.exists()}")
print(f"Validacao oficial:                  {len(val_ofc_ids)} | existe: {VAL_RAW.exists()}")

Treino rotulado (With-GroundTruth): 1351 | existe: True
Validacao oficial:                  451 | existe: True


In [4]:
MODS = ["t1n", "t1c", "t2w", "t2f"]
FILE_ENDING = ".nii.gz"

def case_dir(split_name, case_id):
    if split_name == "train":
        return TRAIN_RAW / case_id
    elif split_name == "val_ofc":
        return VAL_RAW / case_id
    return None

def find_file(folder, key):
    for cand in [folder / f"{key}.nii.gz", folder / f"{key}.nii"]:
        if cand.exists():
            return cand
    cands = sorted(list(folder.glob(f"*{key}*.nii*")))
    if key == "t1n":
        cands = [c for c in cands if "t1c" not in c.name.lower()]
    if not cands:
        raise FileNotFoundError(f"{key} not found in {folder}")
    return cands[0]

def save_as_niigz(src_path, dst_path):
    if src_path.name.lower().endswith(".nii.gz"):
        shutil.copy2(src_path, dst_path)
        return
    nib.save(nib.load(str(src_path)), str(dst_path))

def save_label_niigz(seg_path, dst_path):
    img = nib.load(str(seg_path))
    data = np.asanyarray(img.dataobj).astype(np.int16)
    nib.save(nib.Nifti1Image(data, img.affine, img.header), str(dst_path))

In [5]:
def export_case(split_name, case_id, to_images, to_labels):
    d = case_dir(split_name, case_id)
    for ch, mod in enumerate(MODS):
        save_as_niigz(find_file(d, mod), to_images / f"{case_id}_{ch:04d}{FILE_ENDING}")
    if to_labels is not None:
        save_label_niigz(find_file(d, "seg"), to_labels / f"{case_id}{FILE_ENDING}")

for cid in tqdm(train_ids, desc="Export treino real -> imagesTr/labelsTr"):
    export_case("train", cid, imagesTr, labelsTr)

for cid in tqdm(val_ofc_ids, desc="Export validacao oficial -> imagesVs"):
    export_case("val_ofc", cid, imagesVs, None)

print(f"imagesTr: {len(list(imagesTr.glob(f'*{FILE_ENDING}')))} | labelsTr: {len(list(labelsTr.glob(f'*{FILE_ENDING}')))}")
print(f"  (labeled-only: {len(train_ids)} reais, sem pseudo)")
print(f"imagesVs: {len(list(imagesVs.glob(f'*{FILE_ENDING}')))}")

Export treino real -> imagesTr/labelsTr:   0%|          | 0/1351 [00:00<?, ?it/s]

Export treino real -> imagesTr/labelsTr:   0%|          | 1/1351 [00:00<04:17,  5.25it/s]

Export treino real -> imagesTr/labelsTr:   0%|          | 2/1351 [00:00<04:00,  5.60it/s]

Export treino real -> imagesTr/labelsTr:   0%|          | 3/1351 [00:00<03:46,  5.94it/s]

Export treino real -> imagesTr/labelsTr:   0%|          | 4/1351 [00:00<03:38,  6.16it/s]

...

Export validacao oficial -> imagesVs:  97%|█████████▋| 438/451 [00:10<00:00, 47.53it/s]

Export validacao oficial -> imagesVs:  98%|█████████▊| 443/451 [00:10<00:00, 46.93it/s]

Export validacao oficial -> imagesVs:  99%|█████████▉| 448/451 [00:10<00:00, 44.82it/s]

Export validacao oficial -> imagesVs: 100%|██████████| 451/451 [00:10<00:00, 43.23it/s]

imagesTr: 5404 | labelsTr: 1351
  (labeled-only: 1351 reais, sem pseudo)
imagesVs: 1804


In [6]:
dataset_json = {
    "channel_names": {"0": "T1n", "1": "T1c", "2": "T2w", "3": "T2f"},
    "labels": {"background": 0, "NCR/NET": 1, "ED": 2, "ET": 3},
    "numTraining": len(list(labelsTr.glob(f"*{FILE_ENDING}"))),
    "file_ending": FILE_ENDING,
}
with open(DATASET_FOLDER / "dataset.json", "w", encoding="utf-8") as f:
    json.dump(dataset_json, f, indent=2, ensure_ascii=False)
DATASET_FOLDER / "dataset.json"

PosixPath('/home/henrique.grande/brats_goat_experiments/models_configs/nnUnet_base_1fold_labeled_deform_aug_agressive/nnUNet_raw/Dataset961_BraTS-GoAT-base-1fold-labeled-deform-aug-agressive/dataset.json')

In [7]:
!nnUNetv2_plan_and_preprocess -d {DATASET_ID} --verify_dataset_integrity -pl {PLANNER} -c {CONFIG} -np 4

Fingerprint extraction...
Dataset961_BraTS-GoAT-base-1fold-labeled-deform-aug-agressive


Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer



####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################



Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer


  0%|                                                  | 0/1351 [00:00<?, ?it/s]

...

100%|██████████████████████████████████████▉| 1348/1351 [21:50<00:02,  1.38it/s]

100%|██████████████████████████████████████▉| 1349/1351 [21:52<00:02,  1.11s/it]

100%|██████████████████████████████████████▉| 1350/1351 [21:53<00:00,  1.05it/s]

100%|███████████████████████████████████████| 1351/1351 [21:54<00:00,  1.03s/it]

100%|███████████████████████████████████████| 1351/1351 [21:54<00:00,  1.03it/s]


In [8]:
preprocessed_dataset = NNUNET_PREP / f"Dataset{DATASET_ID:03d}_{DATASET_NAME}"
plans_files = list(preprocessed_dataset.glob(f"{PLANS_NAME}*.json"))
print("Preprocessed:", preprocessed_dataset, "| existe:", preprocessed_dataset.exists())
print("Plans (L):", [p.name for p in plans_files])

Preprocessed: /home/henrique.grande/brats_goat_experiments/models_configs/nnUnet_base_1fold_labeled_deform_aug_agressive/nnUNet_preprocessed/Dataset961_BraTS-GoAT-base-1fold-labeled-deform-aug-agressive | existe: True
Plans (L): ['nnUNetResEncUNetLPlans.json']


In [9]:
# Split de 1 fold com APENAS 5% para validacao (KFold n_splits=20 -> val = 1/20 = 5%; fold 0).
# LABELED-ONLY: nenhum pseudo no treino (mesmo split do baseline Student Labeled Data Only).
real_sorted = sorted(train_ids)
kf = KFold(n_splits=20, shuffle=True, random_state=12345)
tr_idx, val_idx = next(iter(kf.split(real_sorted)))   # fold 0 (95% train / 5% val)
tr_real  = [real_sorted[i] for i in tr_idx]
val_real = [real_sorted[i] for i in val_idx]

splits = [{"train": sorted(tr_real), "val": sorted(val_real)}]
with open(preprocessed_dataset / "splits_final.json", "w", encoding="utf-8") as f:
    json.dump(splits, f, indent=2, ensure_ascii=False)

print(f"train={len(splits[0]['train'])} rotulados (labeled-only) | val={len(splits[0]['val'])} reais (~5%)")

train=1283 rotulados (labeled-only) | val=68 reais (~5%)


In [10]:
# Trainer custom: nnUNetTrainer_deformAug_agressive (augmentations de deformacao + pipeline padrao da nnU-Net)

_DEFORM_SRC = r'''
import numpy as np
import torch
from scipy.ndimage import gaussian_filter, map_coordinates, binary_dilation, center_of_mass
from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer
from batchgeneratorsv2.transforms.base.basic_transform import BasicTransform


# ----------------------- helpers (numpy/scipy, rodam SO no ROI do tumor) -----------------------

def _rand32(shape, rng):
    return rng.standard_normal(shape, dtype=np.float32)


def _smooth_disp(shape, sigma, max_disp, rng):
    nd = len(shape)
    disp = np.empty((nd, *shape), dtype=np.float32)
    for d in range(nd):
        f = gaussian_filter(_rand32(shape, rng), sigma)
        disp[d] = f * np.float32(max_disp / (np.abs(f).max() + 1e-8))
    return disp


def _tumor_weight(seg_bin, dil, ssig):
    w = binary_dilation(seg_bin, iterations=int(dil)).astype(np.float32)
    w = gaussian_filter(w, ssig)
    w /= (w.max() + 1e-8)
    return w


def _tumor_elastic_disp(shape, seg_bin, sigma, max_disp, dil, ssig, rng):
    disp = _smooth_disp(shape, sigma, max_disp, rng)
    disp *= _tumor_weight(seg_bin, dil, ssig)[None]
    return disp


def _tumor_scale_disp(shape, seg_bin, scale, dil, ssig):
    w = _tumor_weight(seg_bin, dil, ssig)
    com = center_of_mass(seg_bin)
    nd = len(shape)
    fac = np.float32(1.0 / scale - 1.0)
    disp = np.empty((nd, *shape), dtype=np.float32)
    for d in range(nd):
        g = (np.arange(shape[d], dtype=np.float32) - np.float32(com[d])) * fac
        sh = [1] * nd; sh[d] = shape[d]
        disp[d] = g.reshape(sh)
    disp *= w[None]
    return disp


def _disp_to_coords(disp):
    # disp (nd, *shape) -> coordenadas absolutas NO SISTEMA DO RECORTE (in-place)
    nd = disp.shape[0]
    shape = disp.shape[1:]
    for d in range(nd):
        g = np.arange(shape[d], dtype=np.float32)
        sh = [1] * nd; sh[d] = shape[d]
        disp[d] += g.reshape(sh)
    return disp


def _tumor_bbox(seg_bin, spatial, margin):
    # bounding-box do tumor (seg>0) + margem, clampado ao patch
    nz = np.where(seg_bin)
    roi = []
    for d in range(len(spatial)):
        lo = max(0, int(nz[d].min()) - margin)
        hi = min(spatial[d], int(nz[d].max()) + 1 + margin)
        roi.append((lo, hi))
    return tuple(roi)


def _edge_fade_mask(shape, fade):
    # janela separavel que e 1 no interior e cai a 0 EXATO nos `fade` voxels de cada borda.
    # forca deslocamento=0 na borda do ROI -> warp do recorte == warp do patch inteiro (sem seam).
    nd = len(shape)
    mask = np.ones(shape, dtype=np.float32)
    for d in range(nd):
        n = shape[d]
        ramp = np.ones(n, dtype=np.float32)
        f = int(min(fade, n // 2))
        if f > 0:
            w = np.linspace(0.0, 1.0, f, dtype=np.float32)  # w[0]=0
            ramp[:f] = w
            ramp[-f:] = w[::-1]
        sh = [1] * nd; sh[d] = n
        mask *= ramp.reshape(sh)
    return mask


# ----------------------- transform -----------------------

class DeformAugTransform(BasicTransform):
    # Deformacao LOCAL ao tumor (tumor_elastic | tumor_scale), restrita ao
    # bounding-box do tumor + margem. Imagem e mascara recebem o MESMO campo.
    # Fora do bbox nada muda (o peso do tumor ja zera o deslocamento la),
    # entao recortar e exatamente equivalente a warpar o patch inteiro.
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self._rng = None

    @property
    def rng(self):
        # lazy: 1 rng por worker (entropia distinta apos o fork; nao recria por amostra)
        if self._rng is None:
            self._rng = np.random.default_rng()
        return self._rng

    def get_parameters(self, **data_dict):
        seg = data_dict["segmentation"]
        spatial = tuple(int(x) for x in seg.shape[1:])
        c = self.cfg
        rng = self.rng

        seg_bin = (seg[0].detach().cpu().numpy() > 0)
        if seg_bin.sum() < c["min_tumor_voxels"] or rng.random() >= c["p_geom"]:
            return {"roi": None, "local_coords": None}

        roi = _tumor_bbox(seg_bin, spatial, c["roi_margin"])
        sl = tuple(slice(a, b) for a, b in roi)
        seg_roi = np.ascontiguousarray(seg_bin[sl])
        shape = seg_roi.shape

        if rng.random() < 0.5:
            disp = _tumor_elastic_disp(shape, seg_roi, c["tumor_elastic_sigma"],
                                       c["tumor_elastic_max_disp"], c["tumor_dilate_iter"],
                                       c["tumor_smooth_sigma"], rng)
        else:
            disp = _tumor_scale_disp(shape, seg_roi, rng.uniform(*c["tumor_scale_range"]),
                                     c["tumor_dilate_iter"], c["tumor_smooth_sigma"])

        disp *= _edge_fade_mask(shape, c["roi_fade"])[None]   # desloc=0 na borda do ROI
        local_coords = _disp_to_coords(disp)
        return {"roi": roi, "local_coords": local_coords}

    def _warp(self, arr, roi, local_coords, order):
        out = arr.copy()
        sl = tuple(slice(a, b) for a, b in roi)
        for ch in range(arr.shape[0]):
            sub = np.ascontiguousarray(arr[(ch, *sl)])
            out[(ch, *sl)] = map_coordinates(sub, local_coords, order=order, mode="nearest")
        return out

    def _apply_to_image(self, img, roi=None, local_coords=None, **params):
        if local_coords is None:
            return img
        arr = img.detach().cpu().numpy()
        out = self._warp(arr, roi, local_coords, order=1)
        return torch.from_numpy(out).to(dtype=img.dtype, device=img.device)

    def _apply_to_segmentation(self, segmentation, roi=None, local_coords=None, **params):
        if local_coords is None:
            return segmentation
        arr = segmentation.detach().cpu().numpy()
        out = self._warp(arr, roi, local_coords, order=0)
        return torch.from_numpy(out).to(dtype=segmentation.dtype, device=segmentation.device)


class DeformAugWrapper(BasicTransform):
    def __init__(self, deform, base):
        super().__init__()
        self.deform = deform
        self.base = base

    def __call__(self, **data_dict):
        data_dict = self.deform(**data_dict)
        return self.base(**data_dict)


_CFG = {
    "p_geom": 0.50,                  # prob. de aplicar deformacao do tumor por amostra
    "min_tumor_voxels": 50,          # abaixo disso, nao deforma
    "roi_margin": 20,                # margem (voxels) alem do bbox do tumor
    "roi_fade": 10,                  # faixa de fade na borda do ROI (>= tumor_elastic_max_disp)
    "tumor_elastic_sigma": 4.0,
    "tumor_elastic_max_disp": 9.0,
    "tumor_dilate_iter": 8,
    "tumor_smooth_sigma": 5.0,
    "tumor_scale_range": (0.65, 1.45),
}


class nnUNetTrainer_deformAug_agressive(nnUNetTrainer):
    @staticmethod
    def get_training_transforms(*args, **kwargs):
        base = nnUNetTrainer.get_training_transforms(*args, **kwargs)
        return DeformAugWrapper(DeformAugTransform(_CFG), base)

'''

_variants_dir = Path(_nnunetv2_pkg.__file__).parent / "training" / "nnUNetTrainer" / "variants"
_tr_file = _variants_dir / "nnUNetTrainer_deformAug_agressive.py"
_tr_file.write_text(_DEFORM_SRC, encoding="utf-8")
_mod_name = "nnunetv2.training.nnUNetTrainer.variants.nnUNetTrainer_deformAug_agressive"
if _mod_name in importlib.sys.modules:
    importlib.reload(importlib.sys.modules[_mod_name])
_mod = importlib.import_module(_mod_name)
assert hasattr(_mod, "nnUNetTrainer_deformAug_agressive"), "Trainer nao registrado."
print("Trainer registrado:", TRAINER)

Trainer registrado: nnUNetTrainer_deformAug_agressive


In [11]:
# Dry-run: aplica a DeformAug num patch sintetico 3D (confirma API + shapes + labels antes de treinar)

_DA = _mod.DeformAugTransform(_mod._CFG)
_img = _t.rand(4, 64, 64, 64)
_seg = _t.zeros(1, 64, 64, 64)
_seg[0, 24:42, 24:42, 24:42] = 1
_seg[0, 30:38, 30:38, 30:38] = 3
_changed = 0
for _ in range(30):
    _o = _DA(image=_img.clone(), segmentation=_seg.clone())
    assert _o["image"].shape == _img.shape and _o["segmentation"].shape == _seg.shape
    assert set(_np.unique(_o["segmentation"].cpu().numpy()).astype(int).tolist()) <= {0, 1, 3}
    if not _t.equal(_o["image"], _img):
        _changed += 1
print(f"Dry-run OK: 30 amostras | shapes preservados | labels in {{0,1,3}} | imagem alterada em {_changed}/30")

Dry-run OK: 30 amostras | shapes preservados | labels in {0,1,3} | imagem alterada em 16/30


In [12]:
# Treino: 1 fold (fold 0), 1000 epocas (nnUNetTrainer padrao), configuracao Large (ResEnc L)
!nnUNetv2_train {DATASET_ID} {CONFIG} {FOLD} -device cuda -p {PLANS_NAME} -tr {TRAINER}

Using device: cuda:0



#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################



2026-07-19 03:59:24.495242: Using torch.compile...


2026-07-19 03:59:25.288125: do_dummy_2d_data_aug: False


2026-07-19 03:59:25.290814: Using splits from existing split file: /home/henrique.grande/brats_goat_experiments/models_configs/nnUnet_base_1fold_labeled_deform_aug_agressive/nnUNet_preprocessed/Dataset961_BraTS-GoAT-base-1fold-labeled-deform-aug-agressive/splits_final.json
2026-07-19 03:59:25.291014: The split file contains 1 splits.
2026-07-19 03:59:25.291040: Desired fold for training: 0
2026-07-19 03:59:25.291059: This split has 1283 training and 68 validation cases.


...

2026-07-20 12:37:46.504123: predicting BraTS-GoAT-02362


2026-07-20 12:37:46.592924: BraTS-GoAT-02362, shape torch.Size([4, 136, 169, 147]), rank 0


2026-07-20 12:37:46.839571: predicting BraTS-GoAT-02483


2026-07-20 12:37:46.908905: BraTS-GoAT-02483, shape torch.Size([4, 139, 178, 142]), rank 0


2026-07-20 12:37:53.234065: Validation complete
2026-07-20 12:37:53.234156: Mean Validation Dice:  0.80902472494788


In [13]:
# Inferencia na VALIDACAO OFICIAL usando o BEST checkpoint
!nnUNetv2_predict -i "{imagesVs}" -o "{PRED_VAL_OFC}" -d {DATASET_ID} -c {CONFIG} -f {FOLD} -p {PLANS_NAME} -tr {TRAINER} -chk checkpoint_best.pth

n_preds = len(list(PRED_VAL_OFC.glob(f"*{FILE_ENDING}")))
print(f"Validacao oficial: {n_preds} mascaras preditas em\n  {PRED_VAL_OFC}")


#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################



There are 451 cases in the source folder
I am processing 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 451 cases that I would like to predict



Predicting BraTS-GoAT-02489:
perform_everything_on_device: True


  0%|                                                     | 0/1 [00:00<?, ?it/s]

100%|█████████████████████████████████████████████| 1/1 [00:01<00:00,  1.15s/it]


...


Predicting BraTS-GoAT-02939:
perform_everything_on_device: True


  0%|                                                     | 0/1 [00:00<?, ?it/s]

100%|█████████████████████████████████████████████| 1/1 [00:00<00:00,  3.64it/s]


sending off prediction to background worker for resampling and export
done with BraTS-GoAT-02939


Validacao oficial: 451 mascaras preditas em
  /home/henrique.grande/brats_goat_experiments/predictions/nnUnet_base_1fold_labeled_deform_aug_agressive/ofc_validation
